# Tugas Mandiri Pertemuan 11: Prediksi Total Penjualan Kopi Nusantara dengan Spark MLlib

**Nama:** Hilmi Mufid | **NPM:** 2505060046 | **Program Studi:** S1 Teknologi Informasi

**Skenario:** Manajemen Kopi Nusantara membutuhkan alat bantu prediksi omzet agar perencanaan stok bahan baku lebih baik. Notebook ini membangun model regresi linear dengan Spark MLlib untuk memprediksi `total_penjualan` sebuah pesanan dari menu, cabang, waktu pesan, jumlah, dan harga satuan.

### Menyiapkan Dataset

Dataset dibuat dengan cell dari modul (2500 baris, `seed=111`) dan disimpan sebagai `tugas11_penjualan.csv`.

In [1]:
import numpy as np
import pandas as pd

np.random.seed(111)
n = 2500
menu_list = ["Kopi Susu", "Americano", "Cappuccino", "Roti Bakar", "Kentang Goreng", "Es Teh"]
cabang_list = ["Magelang", "Yogyakarta", "Semarang"]
waktu_list = ["Pagi", "Siang", "Sore", "Malam"]

data_tugas11 = pd.DataFrame({
    "nama_menu": np.random.choice(menu_list, size=n),
    "cabang": np.random.choice(cabang_list, size=n),
    "waktu_pesan": np.random.choice(waktu_list, size=n),
    "qty": np.random.randint(1, 6, size=n),
    "harga_satuan": np.random.choice([15000, 18000, 20000, 22000, 25000, 30000], size=n),
})
data_tugas11["total_penjualan"] = data_tugas11["qty"] * data_tugas11["harga_satuan"] + np.random.normal(0, 8000, size=n)
data_tugas11.to_csv("tugas11_penjualan.csv", index=False)

print(f"Dataset berhasil dibuat: {len(data_tugas11)} baris")
print(data_tugas11.head())

Dataset berhasil dibuat: 2500 baris
        nama_menu      cabang waktu_pesan  qty  harga_satuan  total_penjualan
0  Kentang Goreng    Semarang       Malam    1         30000     27750.163532
1  Kentang Goreng  Yogyakarta        Sore    3         20000     64216.160744
2  Kentang Goreng    Semarang       Siang    5         22000    108722.792592
3  Kentang Goreng  Yogyakarta        Pagi    2         20000     54483.205872
4      Roti Bakar    Magelang       Malam    2         30000     58157.944120


Membuat `SparkSession`, lalu membaca `tugas11_penjualan.csv`:

In [3]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import col
from pyspark.ml.feature import StringIndexer, VectorAssembler
from pyspark.ml.regression import LinearRegression
from pyspark.ml.evaluation import RegressionEvaluator
import os

spark = SparkSession.builder \
    .appName("Tugas11-MLlibKopiNusantara") \
    .master("local[*]") \
    .getOrCreate()
spark.sparkContext.setLogLevel("ERROR")
spark.conf.set("spark.sql.shuffle.partitions", "4")

print("SparkSession siap. Versi Spark:", spark.version)

# Dataset dibaca dari disk lokal (awalan file:// mencegah Spark mencarinya di HDFS)
path_csv = "file://" + os.path.abspath("tugas11_penjualan.csv")
df_tugas = spark.read.csv(path_csv, header=True, inferSchema=True)

print("Jumlah baris:", df_tugas.count())
df_tugas.printSchema()

SparkSession siap. Versi Spark: 3.5.9
Jumlah baris: 2500
root
 |-- nama_menu: string (nullable = true)
 |-- cabang: string (nullable = true)
 |-- waktu_pesan: string (nullable = true)
 |-- qty: integer (nullable = true)
 |-- harga_satuan: integer (nullable = true)
 |-- total_penjualan: double (nullable = true)



## A. Feature Engineering

`StringIndexer` mengubah `nama_menu`, `cabang`, dan `waktu_pesan` menjadi indeks numerik. Model hasil `fit()` disimpan ke variabel agar dapat dipakai ulang pada bagian E. Setelah itu seluruh fitur (`qty`, `harga_satuan`, dan ketiga kolom indeks) digabung menjadi satu kolom `features` dengan `VectorAssembler`.

In [4]:
# Model StringIndexer disimpan (hasil fit) agar bisa dipakai ulang persis sama pada bagian E
indexer_menu = StringIndexer(inputCol="nama_menu", outputCol="nama_menu_idx").fit(df_tugas)
indexer_cabang = StringIndexer(inputCol="cabang", outputCol="cabang_idx").fit(df_tugas)
indexer_waktu = StringIndexer(inputCol="waktu_pesan", outputCol="waktu_pesan_idx").fit(df_tugas)

df_indexed = indexer_waktu.transform(indexer_cabang.transform(indexer_menu.transform(df_tugas)))

# Seluruh fitur digabung menjadi satu kolom features
assembler = VectorAssembler(
    inputCols=["qty", "harga_satuan", "nama_menu_idx", "cabang_idx", "waktu_pesan_idx"],
    outputCol="features"
)
df_final = assembler.transform(df_indexed)

# Urutan label menunjukkan pemetaan indeks (berdasarkan frekuensi kemunculan)
print("Urutan label nama_menu  :", indexer_menu.labels)
print("Urutan label cabang     :", indexer_cabang.labels)
print("Urutan label waktu_pesan:", indexer_waktu.labels)

df_final.select("nama_menu", "nama_menu_idx", "cabang", "cabang_idx", "waktu_pesan", "waktu_pesan_idx", "features").show(5, truncate=False)

Urutan label nama_menu  : ['Americano', 'Kopi Susu', 'Cappuccino', 'Es Teh', 'Roti Bakar', 'Kentang Goreng']
Urutan label cabang     : ['Magelang', 'Yogyakarta', 'Semarang']
Urutan label waktu_pesan: ['Pagi', 'Malam', 'Siang', 'Sore']
+--------------+-------------+----------+----------+-----------+---------------+-------------------------+
|nama_menu     |nama_menu_idx|cabang    |cabang_idx|waktu_pesan|waktu_pesan_idx|features                 |
+--------------+-------------+----------+----------+-----------+---------------+-------------------------+
|Kentang Goreng|5.0          |Semarang  |2.0       |Malam      |1.0            |[1.0,30000.0,5.0,2.0,1.0]|
|Kentang Goreng|5.0          |Yogyakarta|1.0       |Sore       |3.0            |[3.0,20000.0,5.0,1.0,3.0]|
|Kentang Goreng|5.0          |Semarang  |2.0       |Siang      |2.0            |[5.0,22000.0,5.0,2.0,2.0]|
|Kentang Goreng|5.0          |Yogyakarta|1.0       |Pagi       |0.0            |[2.0,20000.0,5.0,1.0,0.0]|
|Roti Bakar    |

## B. Train/Test Split dan Pelatihan Model

Data dibagi 75% training dan 25% testing (`seed=101`), lalu model `LinearRegression` dilatih dengan `labelCol="total_penjualan"`.

In [5]:
train, test = df_final.randomSplit([0.75, 0.25], seed=101)
print(f"Jumlah data training: {train.count()}")
print(f"Jumlah data testing : {test.count()}")

lr = LinearRegression(featuresCol="features", labelCol="total_penjualan")
model_utama = lr.fit(train)

print("Model berhasil dilatih!")
print("Koefisien tiap fitur:", model_utama.coefficients)
print("Intercept:", model_utama.intercept)

Jumlah data training: 1839
Jumlah data testing : 661
Model berhasil dilatih!
Koefisien tiap fitur: [22046.663271025758,3.022920868160672,-123.21425598657399,279.28452461677676,339.6387953231708]
Intercept: -66903.17766485889


## C. Evaluasi Model

RMSE dan R² dihitung pada data testing.

In [6]:
predictions = model_utama.transform(test)

evaluator_rmse = RegressionEvaluator(labelCol="total_penjualan", predictionCol="prediction", metricName="rmse")
evaluator_r2 = RegressionEvaluator(labelCol="total_penjualan", predictionCol="prediction", metricName="r2")

rmse_utama = evaluator_rmse.evaluate(predictions)
r2_utama = evaluator_r2.evaluate(predictions)

print(f"RMSE : Rp {rmse_utama:,.0f}")
print(f"R²   : {r2_utama:.4f}  ({r2_utama*100:.1f}% variasi data berhasil dijelaskan model)")

RMSE : Rp 10,901
R²   : 0.9107  (91.1% variasi data berhasil dijelaskan model)


**Interpretasi:** R² sekitar 0,9 berarti model berhasil menjelaskan sebagian besar variasi total penjualan, dan RMSE sekitar Rp10.000 berarti prediksi rata rata meleset sekitar sepuluh ribu rupiah dari nilai sebenarnya, sementara rata rata total penjualan per pesanan sekitar Rp65.000. Model ini cukup baik untuk perkiraan kasar omzet dan perencanaan stok, tetapi kurang tepat untuk pesanan tunggal, terutama yang jumlah atau harganya ekstrem (lihat bagian E).

## D. Eksperimen Perbandingan Fitur

Model kedua hanya memakai `qty` dan `harga_satuan`, dilatih dan diuji pada split yang sama dengan model utama.

In [7]:
# Model kedua: HANYA qty dan harga_satuan, pada split training dan testing yang sama
assembler_sederhana = VectorAssembler(inputCols=["qty", "harga_satuan"], outputCol="features_sederhana")
train_s = assembler_sederhana.transform(train)
test_s = assembler_sederhana.transform(test)

lr_sederhana = LinearRegression(featuresCol="features_sederhana", labelCol="total_penjualan")
model_sederhana = lr_sederhana.fit(train_s)
pred_sederhana = model_sederhana.transform(test_s)

rmse_sederhana = evaluator_rmse.evaluate(pred_sederhana)
r2_sederhana = evaluator_r2.evaluate(pred_sederhana)

print(f"Model utama (5 fitur)       : RMSE Rp {rmse_utama:,.0f} | R² {r2_utama:.4f}")
print(f"Model kedua (qty + harga)   : RMSE Rp {rmse_sederhana:,.0f} | R² {r2_sederhana:.4f}")
print(f"Selisih R² (utama dikurangi kedua): {r2_utama - r2_sederhana:+.4f}")

Model utama (5 fitur)       : RMSE Rp 10,901 | R² 0.9107
Model kedua (qty + harga)   : RMSE Rp 10,896 | R² 0.9108
Selisih R² (utama dikurangi kedua): -0.0001


**Kesimpulan:** R² model kedua hampir identik dengan model utama (selisihnya jauh di bawah 0,01, dan bisa positif maupun negatif). Jadi menambahkan informasi menu, cabang, dan waktu pesan **tidak meningkatkan akurasi secara berarti**. Hal ini masuk akal karena `total_penjualan` pada dataset ini dirancang bergantung pada `qty` dan `harga_satuan`, sedangkan menu, cabang, dan waktu pesan tidak ikut menentukan nilainya. Menambah fitur yang tidak relevan tidak membantu, dan indeks hasil `StringIndexer` bahkan memperlakukan kategori seolah memiliki urutan numerik yang sebenarnya tidak ada.

## E. Prediksi Skenario Baru

Tiga skenario pesanan dummy diproses memakai `StringIndexer` yang sudah di-`fit()` pada bagian A, `VectorAssembler` yang sama, dan model utama.

In [9]:
# 3 skenario pesanan baru (data dummy)
skenario = spark.createDataFrame(pd.DataFrame({
    "nama_menu": ["Kopi Susu", "Roti Bakar", "Es Teh"],
    "cabang": ["Magelang", "Yogyakarta", "Semarang"],
    "waktu_pesan": ["Pagi", "Siang", "Malam"],
    "qty": [2, 5, 1],
    "harga_satuan": [22000, 30000, 15000],
}))

# Memakai StringIndexer, VectorAssembler, dan model regresi yang SAMA dengan data training
skenario_idx = indexer_waktu.transform(indexer_cabang.transform(indexer_menu.transform(skenario)))
skenario_final = assembler.transform(skenario_idx)
hasil_skenario = model_utama.transform(skenario_final)

# Kolom qty_x_harga (tanpa variasi acak) ditampilkan sebagai pembanding kasar
hasil_skenario.withColumn("qty_x_harga", col("qty") * col("harga_satuan")) \
    .select("nama_menu", "cabang", "waktu_pesan", "qty", "harga_satuan", "prediction", "qty_x_harga") \
    .show(truncate=False)

+----------+----------+-----------+---+------------+------------------+-----------+
|nama_menu |cabang    |waktu_pesan|qty|harga_satuan|prediction        |qty_x_harga|
+----------+----------+-----------+---+------------+------------------+-----------+
|Kopi Susu |Magelang  |Pagi       |2  |22000       |43571.19372074085 |44000      |
|Roti Bakar|Yogyakarta|Siang      |5  |30000       |134483.46982640692|150000     |
|Es Teh    |Semarang  |Malam      |1  |15000       |1015.8637051739497|15000      |
+----------+----------+-----------+---+------------+------------------+-----------+



**Catatan:** prediksi dekat dengan perkalian `qty x harga_satuan` pada skenario ke 1, tetapi meleset cukup jauh pada skenario ke 2 (pesanan besar, model memprediksi terlalu rendah) dan ke 3 (pesanan satu porsi berharga rendah, model memprediksi mendekati nol). Regresi linear hanya menjumlahkan pengaruh `qty` dan `harga_satuan`, padahal hubungan sebenarnya adalah perkalian keduanya.

## F. Tambahan (di luar rubrik): Fitur Interaksi

Untuk menguji penjelasan di atas, ditambahkan fitur `qty_x_harga` pada split yang sama.

In [10]:
# Fitur interaksi qty x harga_satuan, diterapkan pada split training dan testing yang sama
def tambah_interaksi(df):
    return df.withColumn("qty_x_harga", col("qty") * col("harga_satuan"))

assembler_interaksi = VectorAssembler(
    inputCols=["qty", "harga_satuan", "qty_x_harga"],
    outputCol="features_interaksi"
)
train_i = assembler_interaksi.transform(tambah_interaksi(train))
test_i = assembler_interaksi.transform(tambah_interaksi(test))

model_interaksi = LinearRegression(featuresCol="features_interaksi", labelCol="total_penjualan").fit(train_i)
pred_interaksi = model_interaksi.transform(test_i)

rmse_interaksi = evaluator_rmse.evaluate(pred_interaksi)
r2_interaksi = evaluator_r2.evaluate(pred_interaksi)

print(f"Model utama (5 fitur)        : RMSE Rp {rmse_utama:,.0f} | R² {r2_utama:.4f}")
print(f"Model dengan fitur interaksi : RMSE Rp {rmse_interaksi:,.0f} | R² {r2_interaksi:.4f}")

Model utama (5 fitur)        : RMSE Rp 10,901 | R² 0.9107
Model dengan fitur interaksi : RMSE Rp 7,909 | R² 0.9530


Akurasi naik jelas setelah fitur perkalian ditambahkan, jauh lebih besar daripada hasil menambah menu, cabang, dan waktu pesan. Pelajaran pentingnya: kualitas model lebih ditentukan oleh **fitur yang tepat** daripada banyaknya fitur.

Menutup `SparkSession`:

In [11]:
spark.stop()
print("SparkSession ditutup.")

SparkSession ditutup.
